In [2]:
import torch
import numpy as np
import re
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

In [3]:
torch.cuda.is_available()

True

In [4]:
df_train = pd.read_excel('../data/민원데이터셋_train.xlsx')
df_test = pd.read_excel('../data/민원데이터셋_test.xlsx')

print(df_train.shape)
print(df_test.shape)
df_train.head(5)

(5050, 5)
(1270, 5)


,민원 요청문장,민원명,담당기관(청),담당부서(과),필요문서
0,이사해서 전입신고 언제까지 해야 하나요. 처리까지 예상 소요시간이 궁금합니다.,전입신고,읍·면·동 행정복지센터,민원행정팀/주민등록팀,신분증; 전입신고서; 임대차계약서(선택)
1,전입신고 점검을 요청드립니다. 추가로 필요한 정보가 있나요? 사진/영상 자료가 있습니다.,전입신고,읍·면·동 행정복지센터,민원행정팀/주민등록팀,신분증; 전입신고서; 임대차계약서(선택)
2,이사해서 전입신고 전입신고는 어디서 하나요.,전입신고,읍·면·동 행정복지센터,민원행정팀/주민등록팀,신분증; 전입신고서; 임대차계약서(선택)
3,이사해서 전입신고 세대주 확인이 필요한가요.,전입신고,읍·면·동 행정복지센터,민원행정팀/주민등록팀,신분증; 전입신고서; 임대차계약서(선택)
4,전입신고 정비를 요청드립니다. 접수 후 진행상황은 어떻게 확인하나요?,전입신고,읍·면·동 행정복지센터,민원행정팀/주민등록팀,신분증; 전입신고서; 임대차계약서(선택)


In [5]:
#컬럼명 변경

df_train.columns = ["text", "complaint", "org", "dept", "docs"]
df_test.columns = ["text", "complaint", "org", "dept", "docs"]

In [6]:
df_train.info()
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5050 entries, 0 to 5049
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       5050 non-null   object
 1   complaint  5050 non-null   object
 2   org        5050 non-null   object
 3   dept       5050 non-null   object
 4   docs       5050 non-null   object
dtypes: object(5)
memory usage: 197.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1270 entries, 0 to 1269
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       1270 non-null   object
 1   complaint  1270 non-null   object
 2   org        1270 non-null   object
 3   dept       1270 non-null   object
 4   docs       1270 non-null   object
dtypes: object(5)
memory usage: 49.7+ KB


In [7]:
# 민원요청 문장 정리
def clean_text(text:str) -> str:
    if not isinstance(text, str):
        return ""

    # 줄바꿈, 탭 제거
    text = text.replace("\n", " ").replace("\t", " ")

    # 앞뒤 공백 제거
    text = text.strip()

    # 연속 공백 -> 한 칸
    text = re.sub(r"\s+", " ", text)

    # 과도한 반복 특수문자 정리
    text = re.sub(r"[!]{2,}", "!", text)
    text = re.sub(r"[?]{2,}", "?", text)
    text = re.sub(r"[.]{2,}", ".", text)

    # ㅋㅋㅋㅋ / ㅎㅎㅎㅎ / ㅠㅠㅠㅠ / ㅜㅜㅜㅜ 같은 반복 축소
    text = re.sub(r"(ㅋ)\1{2,}", r"ㅋㅋ", text)
    text = re.sub(r"(ㅎ)\1{2,}", r"ㅎㅎ", text)
    text = re.sub(r"(ㅠ)\1{2,}", r"ㅠㅠ", text)
    text = re.sub(r"(ㅜ)\1{2,}", r"ㅜㅜ", text)

    return text

# 문자열 표준화
def normalize_label(x: str) -> str:
    if not isinstance(x, str):
        return ""
    x = x.strip()
    x = re.sub(r"\s+", " ", x)
    return x

# 필요문서 파싱
def split_docs(doc_str: str):
    if not isinstance(doc_str, str) or doc_str.strip() == "":
        return []

    parts = doc_str.split(";")
    parts = [normalize_label(p) for p in parts]
    parts = [p for p in parts if p != ""]
    return parts

In [8]:
#전처리 적용
df_train["text"] = df_train["text"].apply(clean_text)
df_train["complaint"] = df_train["complaint"].apply(normalize_label)
df_train["org"] = df_train["org"].apply(normalize_label)
df_train["dept"] = df_train["dept"].apply(normalize_label)
df_train["doc_list"] = df_train["docs"].apply(split_docs)

df_test["text"] = df_test["text"].apply(clean_text)
df_test["complaint"] = df_test["complaint"].apply(normalize_label)
df_test["org"] = df_test["org"].apply(normalize_label)
df_test["dept"] = df_test["dept"].apply(normalize_label)
df_test["doc_list"] = df_test["docs"].apply(split_docs)

In [9]:
# 멀티레이블용 labels 리스트 생성
def make_labels(row):
    labels = []

    if row["complaint"]:
        labels.append(f"민원명:{row['complaint']}")
    if row["org"]:
        labels.append(f"기관:{row['org']}")
    if row["dept"]:
        labels.append(f"부서:{row['dept']}")

    for d in row["doc_list"]:
        labels.append(f"문서:{d}")

    # 중복 제거 + 정렬(재현성)
    labels = sorted(list(set(labels)))
    return labels

df_train["labels"] = df_train.apply(make_labels, axis=1)

print(df_train[["text", "labels"]].head(3))

                                                text  \
0        이사해서 전입신고 언제까지 해야 하나요. 처리까지 예상 소요시간이 궁금합니다.   
1  전입신고 점검을 요청드립니다. 추가로 필요한 정보가 있나요? 사진/영상 자료가 있습니다.   
2                           이사해서 전입신고 전입신고는 어디서 하나요.   

                                              labels  
0  [기관:읍·면·동 행정복지센터, 문서:신분증, 문서:임대차계약서(선택), 문서:전입...  
1  [기관:읍·면·동 행정복지센터, 문서:신분증, 문서:임대차계약서(선택), 문서:전입...  
2  [기관:읍·면·동 행정복지센터, 문서:신분증, 문서:임대차계약서(선택), 문서:전입...  


In [10]:
df_test["labels"] = df_test.apply(make_labels, axis=1)

In [11]:
#train valied 데이터 나누기
train_df, val_df = train_test_split(
    df_train,
    test_size=0.2,
    random_state=42,
    stratify=df_train["complaint"]
)

In [12]:
test_df = df_test

In [13]:
#라벨 백터화
mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(train_df["labels"])
y_val = mlb.transform(val_df["labels"])
y_test = mlb.transform(test_df["labels"])

print("전체 라벨 수:", len(mlb.classes_))
print("예시 라벨들:", mlb.classes_[50:70])

전체 라벨 수: 192
예시 라벨들: ['문서:본인인증(온라인) 또는 신분증(방문)' '문서:부모 신분증' '문서:분실 경위(간단)' '문서:불법광고물 사진'
 '문서:사망신고서' '문서:사망진단서/검안서' '문서:사진(해당)' '문서:사회보장급여 신청서' '문서:설치 위치 설명'
 '문서:소득·재산 관련 서류(해당)' '문서:소득·재산 확인 서류(해당)' '문서:소득·재산·가구 정보 서류(해당)'
 '문서:소음 유형' '문서:수수료' '문서:수수료(해당 시)' '문서:시설물 위치' '문서:신고인 신분증' '문서:신분증'
 '문서:신분증 또는 본인인증' '문서:신분증(대체수단)']


c:\Source\TEAM-PJ-DEEP\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:1007: UserWarning: unknown class(es) ['문서:결제정보', '문서:녹음/기록(해당 시)', '문서:민원내용', '문서:방문기록(해당 시)', '문서:배출신청서', '문서:사망진단서', '문서:사진자료', '문서:사진자료(해당 시)', '문서:소득·재산서류', '문서:소득서류', '문서:신고내용', '문서:신고서', '문서:양도증명서', '문서:여권사진', '문서:위임장(대리 시)', '문서:위치정보', '문서:임대차계약서(해당 시)', '문서:증빙자료(해당 시)', '문서:증상정보', '문서:증인정보', '문서:진단서', '문서:판결문/확인서', '문서:품목정보', '문서:허가결정문'] will be ignored
  warnings.warn(


In [14]:
print(y_train)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [1 0 0 ... 0 0 1]
 [0 0 1 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [15]:
label_cols = list(mlb.classes_)

train_label_df = pd.DataFrame(y_train, columns=label_cols, index=train_df.index)
val_label_df = pd.DataFrame(y_val, columns=label_cols, index=val_df.index)
test_label_df = pd.DataFrame(y_test, columns=label_cols, index=test_df.index)

train_out = pd.concat(
    [train_df[["text", "complaint", "org", "dept", "docs", "labels"]], train_label_df],
    axis=1
).reset_index(drop=True)

valid_out = pd.concat(
    [val_df[["text", "complaint", "org", "dept", "docs", "labels"]], val_label_df],
    axis=1
).reset_index(drop=True)

test_out = pd.concat(
    [test_df[["text", "complaint", "org", "dept", "docs", "labels"]], test_label_df],
    axis=1
).reset_index(drop=True)

In [16]:
len(label_cols)

192

In [17]:
train_out.to_csv("../data/train_multilabel.csv", index=False, encoding="utf-8-sig")
valid_out.to_csv("../data/valid_multilabel.csv", index=False, encoding="utf-8-sig")
test_out.to_csv("../data/test_multilabel.csv", index=False, encoding="utf-8-sig")

# 라벨 클래스 저장
with open("../data/multilabel_classes.json", "w", encoding="utf-8") as f:
    json.dump(label_cols, f, ensure_ascii=False, indent=2)

# labels만 간단히 저장한 버전도 같이 저장
train_df[["text", "labels"]].to_json("../data/train_labels_only.json", force_ascii=False, orient="records", indent=2)
val_df[["text", "labels"]].to_json("../data/valid_labels_only.json", force_ascii=False, orient="records", indent=2)
test_df[["text", "labels"]].to_json("../data/test_labels_only.json", force_ascii=False, orient="records", indent=2)

